In [5]:
import os
import pandas as pd
import numpy as np
from scipy import stats

from ML_xgboost import train_xgboost_model
from ML_lightgbm import train_lightgbm_model
from ML_catboost import train_catboost_model
from ML_randomforest import train_randomforest_model

TEST_SIZE = 0.20
VAL_SIZE = 0.20
SEED_LIST = [42, 123, 256, 512, 777, 1024, 2024, 2025,
             3407, 6666, 8888, 10086, 16384, 20516, 27182, 31415]
OUTPUT_DIR = "D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/03_Output/model_stability_depression"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load data
df_model = pd.read_csv("D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/01_Input/df_model.csv")

selected_vars = [
    "somatic_y1", "BMI_T1_cat", "sleep_dura_T1_cat", "sleep_quali_T1", "insomnia_y1",
    "life_satis_y1", "ms_ses_y1", "per_stress_y1", "ms_stress_y1", "depression_y1", "anxiety_y1",
    "ace", "loneliness_y1", "support_y1", "gender_T1", "age_T1", "residence", "income",
    "income_ineqCity_y1", "sss_now", "marrige_par_bin", "edu_pa",
    "eat_unctl_y1", "eat_emot_y1", "food_sweetdrink_T1", "food_takeout_T1",
    "IPAQ_T1_1_bin", "IPAQ_T1_3_bin", "IPAQ_T1_5_bin", "screenT_weekday_T1", "screenT_weekend_T1",
    "psmu_y1", "media_BadMood_T1", "media_GoodMood_T1", "edu_self", "grade_T1"
]

y_col = "depression_y2"
y_cont = pd.to_numeric(df_model[y_col], errors="coerce")
y = (y_cont >= 5).astype(int)
X = df_model[selected_vars].copy()

print(f"Sample size: {len(X)}")
print(f"Positive class rate: {y.mean():.4f}")


# 2. Model registry
model_trainers = {
    "XGBoost": train_xgboost_model,
    "LightGBM": train_lightgbm_model,
    "CatBoost": train_catboost_model,
    "RandomForest": train_randomforest_model
}

# 3. Functions
def calc_ci(series, confidence=0.95):
    """
    Calculate mean, SD, SE, and confidence interval.
    """
    series = pd.Series(series).dropna()
    n = len(series)
    mean_val = series.mean()
    sd_val = series.std(ddof=1) if n > 1 else 0.0
    se_val = sd_val / np.sqrt(n) if n > 1 else 0.0

    if n > 1:
        t_value = stats.t.ppf((1 + confidence) / 2, df=n - 1)
        ci_low = mean_val - t_value * se_val
        ci_high = mean_val + t_value * se_val
    else:
        ci_low, ci_high = mean_val, mean_val

    return {
        "n_runs": n,
        "mean": mean_val,
        "sd": sd_val,
        "se": se_val,
        "ci_low": ci_low,
        "ci_high": ci_high
    }


def summarize_metric(df, group_cols, metric_col):
    """
    Summarize one metric by group.
    """
    summary_rows = []
    grouped = df.groupby(group_cols)

    for group_name, group_df in grouped:
        stats_dict = calc_ci(group_df[metric_col])

        if not isinstance(group_name, tuple):
            group_name = (group_name,)

        row = dict(zip(group_cols, group_name))
        row.update({
            "Metric": metric_col,
            "Mean": stats_dict["mean"],
            "SD": stats_dict["sd"],
            "SE": stats_dict["se"],
            "CI95_Low": stats_dict["ci_low"],
            "CI95_High": stats_dict["ci_high"],
            "N_runs": stats_dict["n_runs"],
            "Mean_SD": f"{stats_dict['mean']:.3f} ± {stats_dict['sd']:.3f}",
            "Mean_CI95": f"{stats_dict['mean']:.3f} ({stats_dict['ci_low']:.3f}, {stats_dict['ci_high']:.3f})"
        })
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


# 4. Multi-seed training
all_results = []

for model_name, trainer in model_trainers.items():
    print("=" * 60)
    print(f"Running model: {model_name}")

    for seed in SEED_LIST:
        print(f"  Seed = {seed}")

        try:
            res = trainer(
                X=X,
                y=y,
                selected_vars=selected_vars,
                random_state=seed,
                test_size=TEST_SIZE,
                val_size=VAL_SIZE
            )

            row_train = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Train",
                "AUC": res.get("train_auc", np.nan),
                "F1": res.get("train_f1", np.nan),
                "Accuracy": res.get("train_acc", np.nan)
            }

            row_val = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Validation",
                "AUC": res.get("val_auc", np.nan),
                "F1": res.get("val_f1", np.nan),
                "Accuracy": res.get("val_acc", np.nan)
            }

            row_test = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Test",
                "AUC": res.get("test_auc", np.nan),
                "F1": res.get("test_f1", np.nan),
                "Accuracy": res.get("test_acc", np.nan)
            }

            all_results.extend([row_train, row_val, row_test])

        except Exception as e:
            print(f"  Error under seed {seed} for {model_name}: {e}")

# 5. Save raw results
df_all_results = pd.DataFrame(all_results)
raw_path = os.path.join(OUTPUT_DIR, "multi_seed_raw_results.csv")
df_all_results.to_csv(raw_path, index=False, encoding="utf-8-sig")

print("\nRaw results saved to:")
print(raw_path)

# 6. Summary results
summary_auc = summarize_metric(df_all_results, ["Model", "Dataset"], "AUC")
summary_f1 = summarize_metric(df_all_results, ["Model", "Dataset"], "F1")
summary_acc = summarize_metric(df_all_results, ["Model", "Dataset"], "Accuracy")

df_summary = pd.concat([summary_auc, summary_f1, summary_acc], axis=0, ignore_index=True)

summary_path = os.path.join(OUTPUT_DIR, "multi_seed_summary_results.csv")
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSummary results saved to:")
print(summary_path)

# 7. Make a wide table
df_summary_for_paper = df_summary.pivot_table(
    index=["Model", "Dataset"],
    columns="Metric",
    values="Mean_SD",
    aggfunc="first"
).reset_index()

paper_table_path = os.path.join(OUTPUT_DIR, "multi_seed_summary_for_paper.csv")
df_summary_for_paper.to_csv(paper_table_path, index=False, encoding="utf-8-sig")

print("\nPaper-style summary table saved to:")
print(paper_table_path)

# 8. Identify the most stable best model on test AUC
test_auc_summary = summary_auc[summary_auc["Dataset"] == "Test"].copy()
test_auc_summary = test_auc_summary.sort_values(by=["Mean", "SD"], ascending=[False, True])

print("\n=== Test AUC summary (sorted by mean desc, SD asc) ===")
print(test_auc_summary[["Model", "Mean", "SD", "CI95_Low", "CI95_High", "Mean_SD", "Mean_CI95"]])

Sample size: 1077
Positive class rate: 0.6045
Running model: XGBoost
  Seed = 42
Start hyperparameter search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits


C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [20:32:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== RandomizedSearchCV ===
Best CV AUC: 0.784243229064201
Best Params: {'xgb__subsample': 0.5, 'xgb__scale_pos_weight': 0.8, 'xgb__reg_lambda': 15, 'xgb__reg_alpha': 5, 'xgb__n_estimators': 1500, 'xgb__min_child_weight': 8, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.02, 'xgb__gamma': 0.1, 'xgb__colsample_bytree': 0.4, 'xgb__colsample_bylevel': 0.5}
[0]	train-logloss:0.67471	eval-logloss:0.67599
[50]	train-logloss:0.61729	eval-logloss:0.62328
[100]	train-logloss:0.58407	eval-logloss:0.59309
[150]	train-logloss:0.56297	eval-logloss:0.57564
[200]	train-logloss:0.54858	eval-logloss:0.56491
[250]	train-logloss:0.53919	eval-logloss:0.55761
[300]	train-logloss:0.53372	eval-logloss:0.55380
[350]	train-logloss:0.52930	eval-logloss:0.55016
[400]	train-logloss:0.52556	eval-logloss:0.54838
[450]	train-logloss:0.52226	eval-logloss:0.54628
[500]	train-logloss:0.51896	eval-logloss:0.54472
[550]	train-logloss:0.51708	eval-logloss:0.54363
[600]	train-logloss:0.51523	eval-logloss:0.54285
[650]	train-

C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [20:32:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[150]	train-logloss:0.51234	eval-logloss:0.59742
[170]	train-logloss:0.51042	eval-logloss:0.59763
  Seed = 256
Start hyperparameter search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== RandomizedSearchCV ===
Best CV AUC: 0.804102318184799
Best Params: {'xgb__subsample': 0.7, 'xgb__scale_pos_weight': 0.5, 'xgb__reg_lambda': 2, 'xgb__reg_alpha': 7, 'xgb__n_estimators': 400, 'xgb__min_child_weight': 5, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.03, 'xgb__gamma': 1.0, 'xgb__colsample_bytree': 0.4, 'xgb__colsample_bylevel': 0.5}
[0]	train-logloss:0.72681	eval-logloss:0.72703
[50]	train-logloss:0.61155	eval-logloss:0.63681
[100]	train-logloss:0.57419	eval-logloss:0.61053
[150]	train-logloss:0.56108	eval-logloss:0.60408
[200]	train-logloss:0.55334	eval-logloss:0.59951
[250]	train-logloss:0.54937	eval-logloss:0.59835


...


=== RandomizedSearchCV Results ===
Best CV AUC: 0.7921
Best Params: {'lgbm__subsample_freq': 1, 'lgbm__subsample': 0.7, 'lgbm__scale_pos_weight': 1, 'lgbm__reg_lambda': 1, 'lgbm__reg_alpha': 10, 'lgbm__objective': 'binary', 'lgbm__num_leaves': 50, 'lgbm__n_estimators': 800, 'lgbm__min_child_samples': 30, 'lgbm__max_depth': 5, 'lgbm__learning_rate': 0.03, 'lgbm__colsample_bytree': 0.6, 'lgbm__boosting_type': 'gbdt'}

Training final model with Early Stopping (Patience=100)...
Training until validation scores don't improve for 100 rounds
[200]	training's binary_logloss: 0.485392	valid_1's binary_logloss: 0.572199
Early stopping, best iteration is:
[106]	training's binary_logloss: 0.503685	valid_1's binary_logloss: 0.570323
  Seed = 31415
Start optimized search on 861 samples with 36 features...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== RandomizedSearchCV Results ===
Best CV AUC: 0.8000
Best Params: {'lgbm__subsample_freq': 1, 'lgbm__subsample': 0.9, 'lgbm__scale_pos_weight': 1, 'lgbm__reg_lambda': 0.1, 'lgbm__reg_alpha': 0.1, 'lgbm__objective': 'binary', 'lgbm__num_leaves': 31, 'lgbm__n_estimators': 800, 'lgbm__min_child_samples': 20, 'lgbm__max_depth': 3, 'lgbm__learning_rate': 0.005, 'lgbm__colsample_bytree': 0.7, 'lgbm__boosting_type': 'gbdt'}

Training final model with Early Stopping (Patience=100)...
Training until validation scores don't improve for 100 rounds
[200]	training's binary_logloss: 0.527891	valid_1's binary_logloss: 0.545464
[400]	training's binary_logloss: 0.473633	valid_1's binary_logloss: 0.511355
[600]	training's binary_logloss: 0.442742	valid_1's binary_logloss: 0.501712
[800]	training's binary_logloss: 0.421663	valid_1's binary_logloss: 0.49878
[1000]	training's binary_logloss: 0.403387	valid_1's binary_logloss: 0.496907
Early stopping, best iteration is:
[1032]	training's binary_logloss: 

C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== CatBoost RandomizedSearchCV ===
Best CV AUC: 0.7806
Best Params: {'cat__subsample': 0.8, 'cat__scale_pos_weight': 5, 'cat__rsm': 0.8, 'cat__learning_rate': 0.01, 'cat__l2_leaf_reg': 3, 'cat__iterations': 500, 'cat__depth': 4}

Training final CatBoost model with Early Stopping...
0:	test: 0.4926839	best: 0.4926839 (0)	total: 2.77ms	remaining: 5.54s
100:	test: 0.8108974	best: 0.8221851 (70)	total: 129ms	remaining: 2.42s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8221850613
bestIteration = 70

Shrink model to first 71 iterations.
  Seed = 123
Start CatBoost hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

=== CatBoost RandomizedSearchCV ===
Best CV AUC: 0.7843
Best Params: {'cat__subsample': 0.7, 'cat__scale_pos_weight': 2, 'cat__rsm': 0.7, 'cat__learning_rate': 0.01, 'cat__l2_leaf_reg': 3, 'cat__iterations': 500, 'cat__depth': 4}

Training final CatBoost model with Early Stopping...
0:	test: 0.6579571	best: 0.6579571 (0

In [ ]:
# RANDOM_STATE = 20516
# TEST_SIZE = 0.20
# VAL_SIZE = 0.20

# # === 1. Training XGBoost ===
# print("=== 1. Training XGBoost ===")
# res_xgb = train_xgboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"XGB Test AUC: {res_xgb['test_auc']:.4f}")

# # === 2. Training LightGBM ===
# print("\n=== 2. Training LightGBM ===")
# res_lgbm = train_lightgbm_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"LGBM Test AUC: {res_lgbm['test_auc']:.4f}")

# # === 3. Training CatBoost ===
# print("\n=== 3. Training CatBoost ===")
# res_cat = train_catboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"CatBoost Test AUC: {res_cat['test_auc']:.4f}")

# # === 4. Training Random Forest ===
# print("\n=== 4. Training RandomForest ===")
# res_rf = train_randomforest_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"RF Test AUC: {res_rf['test_auc']:.4f}")

## 带误差条的性能比较图

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

# === Load data from CSV ===
csv_path = r"D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_depression\multi_seed_summary_results.csv"
df_raw = pd.read_csv(csv_path)

# Normalize model names
df_raw['Model'] = df_raw['Model'].replace({
    'RandomFo': 'RF',
    'RandomForest': 'RF',
    'LightGBM': 'LGBM',
})

# Build long-form df for mean and SD
MODEL_ORDER = ['CatBoost', 'LGBM', 'RF', 'XGBoost']
DATASET_ORDER = ['Train', 'Validation', 'Test']

df_raw['Model'] = pd.Categorical(df_raw['Model'], categories=MODEL_ORDER, ordered=True)
df_raw['Dataset'] = pd.Categorical(df_raw['Dataset'], categories=DATASET_ORDER, ordered=True)
df_raw = df_raw.sort_values(['Metric', 'Model', 'Dataset'])

# === Plot settings (unchanged) ===
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=600, tight_layout=True)

METRICS = ['AUC', 'F1', 'Accuracy']
PALETTES = ['Greens', 'Oranges', 'Purples']
TITLES = ['AUC Performance Comparison', 'F1 Score Performance Comparison', 'Accuracy Performance Comparison']
YLABELS = ['AUC Score', 'F1 Score', 'Accuracy Score']
ALPHA = 0.75  # bar transparency

for ax, metric, palette_name, title, ylabel in zip(axes, METRICS, PALETTES, TITLES, YLABELS):
    sub = df_raw[df_raw['Metric'] == metric].copy()

    n_models = len(MODEL_ORDER)
    n_datasets = len(DATASET_ORDER)
    bar_width = 0.35          # width of each individual bar
    group_width = bar_width * n_datasets   # no gap: groups touch each other internally
    between_gap = 0.30        # gap between model groups

    # x center for each model group
    group_positions = np.arange(n_models) * (group_width + between_gap)
    # offsets within group: bars touch each other (no gap)
    offsets = np.array([-bar_width, 0, bar_width])

    # Get palette colors
    cmap = plt.get_cmap(palette_name)
    # Sample 3 evenly spaced colors from the colormap (avoid too light/too dark)
    colors = [cmap(v) for v in [0.35, 0.6, 0.85]]

    for di, (dataset, color) in enumerate(zip(DATASET_ORDER, colors)):
        means, sds = [], []
        for model in MODEL_ORDER:
            row = sub[(sub['Model'] == model) & (sub['Dataset'] == dataset)]
            means.append(float(row['Mean'].values[0]))
            sds.append(float(row['SD'].values[0]))

        x_pos = group_positions + offsets[di]

        # Draw bars with alpha (半透明)
        ax.bar(
            x_pos, means,
            width=bar_width,          # exactly bar_width, no gap
            color=color,
            alpha=ALPHA,
            label=dataset,
            zorder=3,
            linewidth=0              # no edge line so bars look seamless
        )

        # Error bars
        ax.errorbar(
            x_pos, means,
            yerr=sds,
            fmt='none',
            ecolor='black',
            elinewidth=1.2,
            capsize=4,
            capthick=1.2,
            zorder=4
        )

    ax.set_ylim(0, 1)
    margin = group_width * 0.6
    ax.set_xlim(group_positions[0] - margin, group_positions[-1] + margin)
    ax.set_xticks(group_positions)
    ax.set_xticklabels(MODEL_ORDER)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', length=0)     # remove x tick marks only
    ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)  # y grid only
    ax.grid(axis='x', visible=False)        # no x grid

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7
    )

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

OUTPUT_DIR = r"D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_depression"
png_save_path = os.path.join(OUTPUT_DIR, "Performance_Comparison_WithErrorBars.png")
plt.savefig(png_save_path, dpi=600, bbox_inches='tight')
plt.close()
print(f"Saved to:\n{png_save_path}")

Saved to:
D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_depression\Performance_Comparison_WithErrorBars_v2.png


In [19]:
results = {
    'CatBoost': res_cat,
    'LGBM': res_lgbm,
    'RF': res_rf,
    'XGBoost': res_xgb
}

data_list = []

for model_name, res in results.items():
    # Train set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Train',
        'AUC': res['train_auc'],
        'F1': res['train_f1'],
        'Accuracy': res['train_acc']
    })
    
    # Evalidation set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Validation',
        'AUC': res['val_auc'],
        'F1': res['val_f1'],
        'Accuracy': res['val_acc']
    })
    
    # Test set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Test',
        'AUC': res['test_auc'],
        'F1': res['test_f1'],
        'Accuracy': res['test_acc']
    })

# DataFrame
df_final_perf = pd.DataFrame(data_list)
EXCEL_DIR = os.path.join(PROJECT_ROOT, "03_Output", "ml_dep", "output")
os.makedirs(EXCEL_DIR, exist_ok=True)
excel_save_path = os.path.join(EXCEL_DIR, "Single_Model_Full_Performance.xlsx")
df_final_perf = df_final_perf.sort_values(by=['Model', 'Dataset'])

df_final_perf.to_excel(excel_save_path, index=False)

print("\n" + "="*50)
print(f"All model performances had been saved in: {excel_save_path}")
print("="*50)
print("\n=== The comparison of model performance ===")
print(df_final_perf)

# === Plot ===
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

plt.rcParams['font.family'] = 'Times New Roman'

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=600, tight_layout=True)

# --- 1. AUC ---
sns.barplot(
    x='Model', 
    y='AUC', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[0], 
    palette="Greens"
)
axes[0].set_ylim(0, 1)
axes[0].set_title('AUC Performance Comparison')
axes[0].set_ylabel('AUC Score')
axes[0].tick_params(axis='x')
axes[0].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 2. F1 Score ---
sns.barplot(
    x='Model', 
    y='F1', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[1], 
    palette="Oranges"
)
axes[1].set_ylim(0, 1)
axes[1].set_title('F1 Score Performance Comparison')
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x')
axes[1].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 3. Accuracy ---
sns.barplot(
    x='Model', 
    y='Accuracy', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[2], 
    palette="Purples"
)
axes[2].set_ylim(0, 1)
axes[2].set_title('Accuracy Performance Comparison')
axes[2].set_ylabel('Accuracy Score')
axes[2].tick_params(axis='x')
axes[2].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# plt.suptitle(f'Single Model Performance Across Train, Validation, and Test Sets', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

png_save_path = os.path.join(
    OUTPUT_DIR,
    "Performance_Comparison_5Models_Full_Comparison.png"
)

plt.savefig(png_save_path, dpi=600, bbox_inches='tight')
plt.close()

print(f"\nAll performance comparison has been saved in:\n{png_save_path}")

NameError: name 'res_cat' is not defined

In [ ]:
import os
import pandas as pd
import shap
import matplotlib.pyplot as plt
from ML_shap_dep import run_full_shap_analysis

VARIABLE_MAPPING = {
    "somatic_y1": "Somatic Symptoms",
    "BMI_T1_cat": "BMI Category", 
    "sleep_dura_T1_cat": "Sleep Duration",
    "sleep_quali_T1": "Sleep Quality",
    "insomnia_y1": "Insomnia",
    
    "life_satis_y1": "Life Satisfaction",
    "ms_ses_y1": "Mindset of Socioeconomic Status", 
    "per_stress_y1": "Perceived Stress", 
    "ms_stress_y1": "Stress Mindset",
    "depression_y1": "Baseline Depressive Symptoms",
    "anxiety_y1": "BaselineAnxiety Symptoms",
    
    "ace": "Adverse Childhood Experiences",
    
    "loneliness_y1": "Loneliness", 
    "support_y1": "Social Support",
    
    "gender_T1": "Gender",
    "age_T1": "Age",
    "grade_T1": "Grade",
    "residence": "Residence", 
    "income": "Household Income",
    "income_ineqCity_y1": "Perceived Income Inequality",
    "sss_now": "Subjective Socialeconomic Status",
    "marrige_par_bin": "Parental Marital Status",
    "edu_pa": "Parental Educational Level",
    
    "eat_unctl_y1": "Uncontrolled Eating",
    "eat_emot_y1": "Emotional Eating", 
    "food_sweetdrink_T1": "Sugar-Sweetened Beverage Consumption",
    "food_takeout_T1": "Takeaway Frequency",
    
    "IPAQ_T1_1_bin": "Vigorous Physical Activity",
    "IPAQ_T1_3_bin": "Moderate Physical Activity", 
    "IPAQ_T1_5_bin": "Walking Activity",
    
    "screenT_weekday_T1": "Weekday Screen Time",
    "screenT_weekend_T1": "Weekend Screen Time",
    
    "psmu_y1": "Problematic Social Media Use",
    "media_BadMood_T1": "Social Media Posting of Negative Emotion", 
    "media_GoodMood_T1": "Social Media Posting of Positive Emotion",
    
    "edu_self": "Educational Aspiration"
}


all_results = {
    'RandomForest': res_rf,
    'XGBoost': res_xgb,
    'LightGBM': res_lgbm,
    'CatBoost': res_cat
}


FLAG_SHOW = False 
FLAG_TITLE = False 
IS_LOG = False 
TOPN = 15


for name, results in all_results.items():
    run_full_shap_analysis(
        model_name=name,
        results=results,
        df_model=df_model,         
        selected_vars=selected_vars,
        variable_mapping=VARIABLE_MAPPING,
        is_log_transformed=IS_LOG,
        top_n=TOPN,
        flag_show=FLAG_SHOW,
        flag_title=FLAG_TITLE
    )